# ContractNLI: measure before building
Checks the format, the labels, how guessable the labels are from the statement alone, and how often a cut-down passage keeps the evidence. Nothing is scored or published.

**First:** check the license on https://stanfordnlp.github.io/contract-nli/ and paste the download link into Cell 2.

## Cell 1 — Install and imports
No GPU needed. Nothing is scored or published by this notebook.

In [ ]:
!pip install -q sentence-transformers
import os, glob, json, random, zipfile, urllib.request
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

## Cell 2 — Settings
Paste the download link from the official page into CONTRACTNLI_URL.

In [ ]:
# Copy this link from https://stanfordnlp.github.io/contract-nli/ (the authors' page), not from a copy.
CONTRACTNLI_URL = "https://stanfordnlp.github.io/contract-nli/resources/contract-nli.zip"
WORK_DIR    = "/content/contractnli"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"      # the same model that picks hard distractors in the builder
BUDGETS     = [1200, 1600, 2000]            # passage lengths to try, in characters
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # bge's recommended query prefix
SEED = 0

## Cell 3 — Download and look at the structure
Print the real layout first; stop with a clear message if it differs.

In [ ]:
os.makedirs(WORK_DIR, exist_ok=True)
zip_path = f"{WORK_DIR}/contractnli.zip"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(CONTRACTNLI_URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(WORK_DIR)
files = sorted(glob.glob(f"{WORK_DIR}/**/*.json", recursive=True))
print("JSON files in the archive:")
for f in files:
    print("  ", f.replace(WORK_DIR + "/", ""), f"({os.path.getsize(f) / 1e6:.1f} MB)")

def find(split_word):
    hits = [f for f in files if os.path.basename(f).lower().startswith(split_word)]
    assert len(hits) == 1, f"expected one file starting with '{split_word}', found {hits}"
    return hits[0]

RAW = {name: json.load(open(find(word))) for name, word in [("train", "train"), ("dev", "dev"), ("test", "test")]}
first = RAW["train"]
print("\ntop-level keys:", list(first.keys()))
doc0 = first["documents"][0]
print("keys of one document:", list(doc0.keys()))
ann0 = doc0["annotation_sets"][0]["annotations"]
k0 = sorted(ann0)[0]
print(f"one annotation ({k0}):", ann0[k0])
print("one hypothesis entry:", k0, "->", first["labels"][k0])
for key in ["documents", "labels"]:
    assert key in first, f"the file has no '{key}' key: the format differs from what this notebook expects"
for key in ["text", "spans", "annotation_sets"]:
    assert key in doc0, f"documents have no '{key}' key: the format differs from what this notebook expects"
print("\nformat as expected")

JSON files in the archive:
   contract-nli/dev.json (1.2 MB)
   contract-nli/test.json (2.3 MB)
   contract-nli/train.json (7.6 MB)

top-level keys: ['documents', 'labels']
keys of one document: ['id', 'file_name', 'text', 'spans', 'annotation_sets', 'document_type', 'url']
one annotation (nda-1): {'choice': 'Entailment', 'spans': [14]}
one hypothesis entry: nda-1 -> {'short_description': 'Explicit identification', 'hypothesis': 'All Confidential Information shall be expressly identified by the Disclosing Party.'}

format as expected


## Cell 4 — Turn the files into rows: one row per (contract, hypothesis)
And the basic numbers: documents, hypotheses, labels.

In [ ]:
HYPS = {k: v["hypothesis"] for k, v in RAW["train"]["labels"].items()}
LABEL_WORDS = {"Entailment": "entailed", "Contradiction": "contradicted", "NotMentioned": "not mentioned"}

def rows_of(raw, split):
    out = []
    for d in raw["documents"]:
        spans = [d["text"][a:b] for a, b in d["spans"]]
        for key, a in d["annotation_sets"][0]["annotations"].items():
            out.append(dict(split=split, doc_id=d["id"], hyp_key=key, hypothesis=HYPS[key],
                            label=LABEL_WORDS[a["choice"]], evidence=list(a["spans"]),
                            spans=spans, doc_chars=len(d["text"])))
    return out

ROWS = {s: rows_of(RAW[s], s) for s in RAW}
print(f"hypotheses: {len(HYPS)}")
for k, h in HYPS.items():
    print(f"  {k:7s} {h}")
print()
summary = pd.DataFrame([{"split": s, "contracts": len(RAW[s]["documents"]), "rows": len(r),
                         **Counter(x["label"] for x in r)} for s, r in ROWS.items()])
print(summary.to_string(index=False))
lens = pd.Series([x["doc_chars"] for x in ROWS["train"][::len(HYPS)]])
print(f"\ncontract length (characters), train: median {lens.median():.0f}, "
      f"90% below {lens.quantile(0.9):.0f}, longest {lens.max():.0f}")

hypotheses: 17
  nda-11  Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
  nda-16  Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.
  nda-15  Agreement shall not grant Receiving Party any right to Confidential Information.
  nda-10  Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.
  nda-2   Confidential Information shall only include technical information.
  nda-1   All Confidential Information shall be expressly identified by the Disclosing Party.
  nda-19  Some obligations of Agreement may survive termination of Agreement.
  nda-12  Receiving Party may independently develop information similar to Confidential Information.
  nda-20  Receiving Party may retain some Confidential Information even after the return or destruction of Confidential Information.
  nda-3   Confidential Information may include verbally conveyed information.


## Cell 5 — Hypothesis-only baseline: how much can be guessed without reading?
Learn each statement's most common label on train, apply it to dev and test.

In [ ]:
by_hyp = defaultdict(Counter)
for x in ROWS["train"]:
    by_hyp[x["hyp_key"]][x["label"]] += 1
majority_by_hyp = {k: c.most_common(1)[0][0] for k, c in by_hyp.items()}
overall_majority = Counter(x["label"] for x in ROWS["train"]).most_common(1)[0][0]

for split in ("dev", "test"):
    rows = ROWS[split]
    acc_hyp = np.mean([x["label"] == majority_by_hyp[x["hyp_key"]] for x in rows])
    acc_all = np.mean([x["label"] == overall_majority for x in rows])
    print(f"{split}: always '{overall_majority}' = {acc_all:.3f} | each statement's usual label = {acc_hyp:.3f}")

per_hyp = pd.DataFrame([{"hypothesis": k, **c, "share of its most common label": c.most_common(1)[0][1] / sum(c.values())}
                        for k, c in sorted(by_hyp.items())]).fillna(0)
print("\nper statement, train:")
print(per_hyp.round(2).to_string(index=False))

dev: always 'entailed' = 0.500 | each statement's usual label = 0.681
test: always 'entailed' = 0.463 | each statement's usual label = 0.659

per statement, train:
hypothesis  entailed  not mentioned  contradicted  share of its most common label
     nda-1        91            220         112.0                            0.52
    nda-10       159            262           2.0                            0.62
    nda-11        59            363           1.0                            0.86
    nda-12       263            160           0.0                            0.62
    nda-13       311            112           0.0                            0.74
    nda-15       327             96           0.0                            0.77
    nda-16       181            241           1.0                            0.57
    nda-17        91            256          76.0                            0.61
    nda-18        93            330           0.0                            0.78
    nda-19      

## Cell 6 — Passage recall: does a cut-down passage still contain the evidence?
Pick the most relevant paragraphs for each statement, the same way for every label.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
embedder = SentenceTransformer(EMBED_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")

hyp_vec = dict(zip(HYPS, embedder.encode([QUERY_PREFIX + h for h in HYPS.values()], normalize_embeddings=True)))
span_vecs = {}                                   # one embedding per paragraph, per contract
for split, rows in ROWS.items():
    for x in rows[::len(HYPS)]:                  # once per contract
        key = (split, x["doc_id"])
        if key not in span_vecs:
            span_vecs[key] = embedder.encode(x["spans"], normalize_embeddings=True, batch_size=64)
print("embedded", len(span_vecs), "contracts")

def select(x, budget):
    """Most similar paragraphs first, until the budget is full; returned in document order."""
    sims = span_vecs[(x["split"], x["doc_id"])] @ hyp_vec[x["hyp_key"]]
    chosen, used = [], 0
    for i in np.argsort(-sims):
        n = len(x["spans"][i]) + 1
        if used + n > budget and chosen:
            continue
        chosen.append(int(i)); used += n
        if used >= budget:
            break
    return sorted(chosen), float(sims.max())

lines, keep = [], defaultdict(Counter)
for budget in BUDGETS:
    for split, rows in ROWS.items():
        for x in rows:
            chosen, _ = select(x, budget)
            if x["label"] == "not mentioned":
                keep[budget, split]["not mentioned"] += 1
                continue
            hit_any = bool(set(chosen) & set(x["evidence"]))
            hit_all = set(x["evidence"]) <= set(chosen)
            lines.append({"budget": budget, "split": split, "label": x["label"], "any evidence kept": hit_any,
                          "all evidence kept": hit_all})
            if hit_any:
                keep[budget, split][x["label"]] += 1
recall = (pd.DataFrame(lines).groupby(["budget", "split", "label"])[["any evidence kept", "all evidence kept"]]
          .mean().round(3).reset_index())
print("\nPASSAGE RECALL (rows with evidence, i.e. entailed or contradicted)")
print(recall.to_string(index=False))

print("\nROWS LEFT after dropping entailed/contradicted rows whose evidence didn't fit")
print(pd.DataFrame([{"budget": b, "split": s, **c, "total": sum(c.values())} for (b, s), c in sorted(keep.items())]
                   ).fillna(0).to_string(index=False))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedded 607 contracts

PASSAGE RECALL (rows with evidence, i.e. entailed or contradicted)
 budget split        label  any evidence kept  all evidence kept
   1200   dev contradicted              0.589              0.263
   1200   dev     entailed              0.728              0.368
   1200  test contradicted              0.705              0.423
   1200  test     entailed              0.751              0.370
   1200 train contradicted              0.698              0.436
   1200 train     entailed              0.734              0.383
   1600   dev contradicted              0.705              0.368
   1600   dev     entailed              0.796              0.424
   1600  test contradicted              0.786              0.491
   1600  test     entailed              0.817              0.434
   1600 train contradicted              0.782              0.511
   1600 train     entailed              0.795              0.444
   2000   dev contradicted              0.789              0.442

## Cell 7 — Shortcut check and two example questions
Do 'not mentioned' passages look different? And what would the student actually read?

In [ ]:
budget = BUDGETS[1]
sim_rows = []
for x in ROWS["train"]:
    chosen, top_sim = select(x, budget)
    sim_rows.append({"label": x["label"], "best paragraph similarity": top_sim})
print(f"best-paragraph similarity by label (train, budget {budget}): if 'not mentioned' is much lower,")
print("a student could partly guess it from how relevant the passage looks")
print(pd.DataFrame(sim_rows).groupby("label")["best paragraph similarity"].describe()[["mean", "25%", "50%", "75%"]].round(3))

rng = random.Random(SEED)
for label in ("entailed", "contradicted"):
    x = rng.choice([r for r in ROWS["train"] if r["label"] == label and set(select(r, budget)[0]) & set(r["evidence"])])
    chosen, _ = select(x, budget)
    passage = " ".join(x["spans"][i] for i in chosen)
    print("\n" + "=" * 70)
    print(f"STATEMENT: {x['hypothesis']}\nRIGHT ANSWER: {x['label']}\nPASSAGE ({len(passage)} chars):\n{passage[:900]}{' ...' if len(passage) > 900 else ''}")

best-paragraph similarity by label (train, budget 1600): if 'not mentioned' is much lower,
a student could partly guess it from how relevant the passage looks
                mean    25%    50%    75%
label                                    
contradicted   0.759  0.728  0.765  0.793
entailed       0.772  0.736  0.775  0.806
not mentioned  0.738  0.701  0.739  0.775

STATEMENT: Receiving Party shall not solicit some of Disclosing Party's representatives.
RIGHT ANSWER: entailed
PASSAGE (1591 chars):
Agreement This Non-Disclosure  Nothing herein shall be deemed to grant to the Recipient a license under the Disclosing Party’s intellectual property rights. Recipient will promptly notify the Disclosing Party of any unauthorized disclosure of Confidential Information or other breaches of this Agreement. Prior to making any such disclosure, the Recipient shall make commercially reasonable efforts to provide the Disclosing Party with: The Parties agree that this Agreement does not require or c